In [ ]:

!pip -q install timm

import os, json, math, random, time
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from PIL import Image
from torchvision import transforms
import timm

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(42)


Using device: cuda


## 1) Mount Google Drive + Set paths


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

DATA_CSV = "/content/drive/MyDrive/AURA/final_3000_dataset/final_3000_correct_mapping.csv"

IMAGE_BASE = ""  # e.g., "/content/drive/MyDrive/CU_dataset"

ARTIFACT_ROOT = "/content/drive/MyDrive/AURA_artifacts/gcmupen_v3_final/weights"
os.makedirs(ARTIFACT_ROOT, exist_ok=True)

print("DATA_CSV:", DATA_CSV)
print("IMAGE_BASE:", IMAGE_BASE)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)


Mounted at /content/drive
DATA_CSV: /content/drive/MyDrive/AURA/final_3000_dataset/final_3000_correct_mapping.csv
IMAGE_BASE: 
ARTIFACT_ROOT: /content/drive/MyDrive/AURA_artifacts/gcmupen_v3_final/weights


## 2) Load CSV + Detect columns


In [3]:

df = pd.read_csv(DATA_CSV)

print("Columns:", df.columns.tolist())
print(df.head(2))

# -------------------------
# Auto-detect key columns
# -------------------------
def pick_first(existing, candidates):
    for c in candidates:
        if c in existing:
            return c
    return None

colset = set(df.columns)

IMAGE_COL = pick_first(colset, ["image_path", "img_path", "image", "path", "filepath"])
DRUG_COL  = pick_first(colset, ["drug_group", "drugclass", "drug_class", "label", "target"])
SPLIT_COL = pick_first(colset, ["split", "Split", "fold", "set"])

print("Detected IMAGE_COL:", IMAGE_COL)
print("Detected DRUG_COL :", DRUG_COL)
print("Detected SPLIT_COL:", SPLIT_COL)

if IMAGE_COL is None or DRUG_COL is None:
    raise ValueError("Could not detect required columns. Please set IMAGE_COL and DRUG_COL manually.")

# If no split column exists, we create a stratified-ish split (simple).
if SPLIT_COL is None:
    print("No split column found -> creating split (80/10/10).")
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    n = len(df)
    train_end = int(0.8*n)
    val_end   = int(0.9*n)
    df["split"] = "train"
    df.loc[train_end:val_end-1, "split"] = "validation"
    df.loc[val_end:, "split"] = "test"
    SPLIT_COL = "split"

print(df[SPLIT_COL].value_counts())

# Ensure drug labels are strings
df[DRUG_COL] = df[DRUG_COL].astype(str)


Columns: ['image_id', 'image_path', 'split', 'patient_id', 'visit_id', 'symptoms_raw', 'diagnosis_raw', 'investigations_raw', 'CRP', 'FT4', 'IgE', 'VitD', 'meds_all_raw', 'Age', '1.History of Chronic Urticaria', 'Sex', 'Weight', 'Height', 'Symptoms Of Urticaria', 'When did you first experience the symptoms of urticaria ', 'Age experienced the first symptoms', 'Diagnosed at the age of', 'Duration of Symptoms of urticaria', 'If Wheals are present', 'The shape of an individual wheal', 'Size of a single Wheal', 'No. of wheals', 'Duration of wheal', 'Itching score', 'Location', 'If  angioedema is present', 'Duration of angioedema', 'Location.1', 'Discomfort of Swelling', 'Affect of Swelling on Daily activities', 'Angioedema affect on appearance', 'Overall affect of Swelling', '1.7 If angioedema is present,\nWhich of the following Drugs do you use', 'Remission of Angioedema after discontinuation of the drug:', 'Which applies to your wheals/angioedema or both?', 'If wheals/angioedema or both 

## 3) Create guideline steps

In [ ]:

import re

DRUG_NAME_COL = None
for c in ["drug_name", "medication", "drug", "prescribed_drug", "treatment", "medicine"]:
    if c in df.columns:
        DRUG_NAME_COL = c
        break

print("Detected DRUG_NAME_COL:", DRUG_NAME_COL)

def map_guideline_step(row):
    group = str(row[DRUG_COL]).upper()
    drugtxt = ""
    if DRUG_NAME_COL is not None:
        drugtxt = str(row[DRUG_NAME_COL]).lower()

    # STEP 4 – Advanced therapy (omalizumab / cyclosporine / biologics)
    if group == "ADVANCED_THERAPY" or re.search(r"\b(omalizumab|cyclosporine)\b", drugtxt):
        return 4

    # STEP 3 – Potent antihistamines (examples from guideline)
    if re.search(r"\b(doxepin|hydroxyzine)\b", drugtxt):
        return 3

    # STEP 2 – Add-ons (montelukast LTRA / H2 blockers / 1st gen bedtime)
    if group == "LTRA" or re.search(r"\b(montelukast)\b", drugtxt):
        return 2
    if re.search(r"\b(famotidine|ranitidine|cimetidine|nizatidine)\b", drugtxt):
        return 2
    if re.search(r"\b(diphenhydramine|chlorpheniramine)\b", drugtxt):
        return 2

    # STEP 1 – Default (modern 2nd gen antihistamine mono-therapy)
    return 1

df["guideline_step"] = df.apply(map_guideline_step, axis=1).astype(int)
df["step_id"] = (df["guideline_step"] - 1).astype(int)  # 0..3

print("Guideline step distribution:")
print(df["guideline_step"].value_counts().sort_index())

assert df["step_id"].min() >= 0 and df["step_id"].max() <= 3


Detected DRUG_NAME_COL: None
Guideline step distribution:
guideline_step
1    2250
2     300
4     450
Name: count, dtype: int64


## 4) Configure features (labs + clinical)


In [5]:

# -------------------------
# Auto-detect numeric feature columns (excluding labels/ids)
# -------------------------
exclude = {IMAGE_COL, DRUG_COL, SPLIT_COL, "guideline_step", "step_id"}
if DRUG_NAME_COL is not None:
    exclude.add(DRUG_NAME_COL)

numeric_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
print("Detected numeric feature columns:", len(numeric_cols))

# Heuristic split: first half as labs, second half as clinical.
# If you know them, replace LAB_COLS and CLIN_COLS manually.
LAB_COLS = numeric_cols[:len(numeric_cols)//2]
CLIN_COLS = numeric_cols[len(numeric_cols)//2:]

print("LAB_COLS:", LAB_COLS[:10], "..." if len(LAB_COLS)>10 else "")
print("CLIN_COLS:", CLIN_COLS[:10], "..." if len(CLIN_COLS)>10 else "")

# Train-only normalization stats (no leakage)
train_df = df[df[SPLIT_COL].astype(str).str.lower().isin(["train"])].copy()
val_df   = df[df[SPLIT_COL].astype(str).str.lower().isin(["validation","val"])].copy()
test_df  = df[df[SPLIT_COL].astype(str).str.lower().isin(["test"])].copy()

if len(val_df)==0:
    # if your split uses "valid"
    val_df = df[df[SPLIT_COL].astype(str).str.lower().isin(["valid"])].copy()

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))

lab_means = train_df[LAB_COLS].mean().values if len(LAB_COLS)>0 else np.array([])
lab_stds  = train_df[LAB_COLS].std().values + 1e-8 if len(LAB_COLS)>0 else np.array([])
clin_means = train_df[CLIN_COLS].mean().values if len(CLIN_COLS)>0 else np.array([])
clin_stds  = train_df[CLIN_COLS].std().values + 1e-8 if len(CLIN_COLS)>0 else np.array([])

print("Lab dims:", len(LAB_COLS), "Clinical dims:", len(CLIN_COLS))


Detected numeric feature columns: 11
LAB_COLS: ['CRP', 'FT4', 'IgE', 'VitD', 'Age'] 
CLIN_COLS: ['Weight', 'Height', 'Age experienced the first symptoms', 'Diagnosed at the age of', 'Itching score', '1.7 If angioedema is present,\nWhich of the following Drugs do you use'] 
Train/Val/Test: 2062 484 454
Lab dims: 5 Clinical dims: 6


## 5) Label vocabularies

In [6]:

drug_groups = sorted(df[DRUG_COL].unique().tolist())
drug2id = {d:i for i,d in enumerate(drug_groups)}
id2drug = {i:d for d,i in drug2id.items()}

num_drugs = len(drug_groups)
num_steps = 4

print("Drug groups:", drug_groups)
print("num_drugs:", num_drugs, "num_steps:", num_steps)


Drug groups: ['ADVANCED_THERAPY', 'H1_ANTIHISTAMINE', 'LTRA', 'OTHER']
num_drugs: 4 num_steps: 4


## 6) Transforms + Dataset

In [7]:

IMG_SIZE = 224

def get_transforms(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])

class CUMuPeNDataset(Dataset):
    def __init__(self, df, transform, image_base=""):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.image_base = image_base
        self.num_steps = num_steps

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, p: str) -> str:
        p = str(p)
        if os.path.isabs(p) or self.image_base == "":
            return p
        return os.path.join(self.image_base, p)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self._resolve_path(row[IMAGE_COL])
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise FileNotFoundError(f"Could not open image: {img_path} | {e}")

        img = self.transform(img)

        # Features
        if len(LAB_COLS)>0:
            lab = (row[LAB_COLS].values.astype(np.float32) - lab_means) / lab_stds
        else:
            lab = np.zeros((0,), dtype=np.float32)

        if len(CLIN_COLS)>0:
            clin = (row[CLIN_COLS].values.astype(np.float32) - clin_means) / clin_stds
        else:
            clin = np.zeros((0,), dtype=np.float32)

        y_drug = drug2id[str(row[DRUG_COL])]
        y_step = int(row["step_id"])
        y_step = max(0, min(y_step, self.num_steps-1))  # safety clamp

        return {
            "image": img,
            "lab_feats": torch.tensor(lab, dtype=torch.float32),
            "clinical_feats": torch.tensor(clin, dtype=torch.float32),
            "drug_label": torch.tensor(y_drug, dtype=torch.long),
            "step_label": torch.tensor(y_step, dtype=torch.long),
        }


## 7) DataLoaders (with balanced sampler)


In [8]:

BATCH_SIZE = 32
NUM_WORKERS = 2

train_ds = CUMuPeNDataset(train_df, transform=get_transforms(True), image_base=IMAGE_BASE)
val_ds   = CUMuPeNDataset(val_df, transform=get_transforms(False), image_base=IMAGE_BASE)
test_ds  = CUMuPeNDataset(test_df, transform=get_transforms(False), image_base=IMAGE_BASE)

# Weighted sampler for drug classes
train_labels = train_df[DRUG_COL].map(drug2id).values
class_counts = np.bincount(train_labels, minlength=num_drugs)
class_weights = 1.0 / (class_counts + 1e-6)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(weights=torch.tensor(sample_weights, dtype=torch.double),
                                num_samples=len(sample_weights),
                                replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print("Train/Val/Test batches:", len(train_loader), len(val_loader), len(test_loader))
print("Train class counts:", class_counts)


Train/Val/Test batches: 65 16 15
Train class counts: [ 306 1015  209  532]


## 8) Model: EfficientNet + FiLM + Gated Fusion (novel + stable)
- Image tower: EfficientNet-B3 (timm)
- Lab tower + Clinical tower
- FiLM: labs modulate image features
- Gating: combines modalities adaptively
- Heads: drug_group + guideline_step


In [9]:
class FiLM(nn.Module):
    def __init__(self, cond_dim: int, feat_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, feat_dim * 2)
        )

    def forward(self, feats, cond):
        gb = self.net(cond)
        gamma, beta = gb.chunk(2, dim=-1)
        return feats * (1 + gamma) + beta

class GC_MuPeN(nn.Module):
    def __init__(self, lab_in_dim: int, clinical_in_dim: int,
                 num_drugs: int, num_steps: int,
                 dropout: float = 0.3, fusion_hidden: int = 512, fusion_out: int = 256):
        super().__init__()

        # Image backbone
        self.image_backbone = timm.create_model("efficientnet_b3", pretrained=True, num_classes=0)
        img_feat_dim = self.image_backbone.num_features

        # Lab / Clinical towers
        self.lab_mlp = nn.Sequential(
            nn.Linear(lab_in_dim, 128) if lab_in_dim>0 else nn.Identity(),
            nn.ReLU() if lab_in_dim>0 else nn.Identity(),
            nn.Dropout(dropout) if lab_in_dim>0 else nn.Identity(),
            nn.Linear(128, 128) if lab_in_dim>0 else nn.Identity()
        )

        self.clin_mlp = nn.Sequential(
            nn.Linear(clinical_in_dim, 128) if clinical_in_dim>0 else nn.Identity(),
            nn.ReLU() if clinical_in_dim>0 else nn.Identity(),
            nn.Dropout(dropout) if clinical_in_dim>0 else nn.Identity(),
            nn.Linear(128, 128) if clinical_in_dim>0 else nn.Identity()
        )

        # FiLM conditioning (labs → image feats)
        cond_dim = 128 if lab_in_dim>0 else 0
        self.film = FiLM(cond_dim, img_feat_dim) if cond_dim>0 else None

        # Fusion
        fused_in_dim = img_feat_dim + (128 if lab_in_dim>0 else 0) + (128 if clinical_in_dim>0 else 0)

        self.fusion = nn.Sequential(
            nn.Linear(fused_in_dim, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, fusion_out),
            nn.ReLU(),
        )

        # Gating (learn modality weights)
        self.gate = nn.Sequential(
            nn.Linear(fused_in_dim, 3),
            nn.Softmax(dim=-1)
        )

        self.drug_head = nn.Linear(fusion_out, num_drugs)
        self.step_head = nn.Linear(fusion_out, num_steps)

        self._freeze_backbone(True)

    def _freeze_backbone(self, freeze: bool):
        for p in self.image_backbone.parameters():
            p.requires_grad = not freeze

    def unfreeze_backbone(self):
        self._freeze_backbone(False)

    def forward(self, image, lab_feats, clinical_feats):
        img_feats = self.image_backbone(image)

        if isinstance(self.lab_mlp, nn.Identity):
            lab_emb = None
        else:
            lab_emb = self.lab_mlp(lab_feats)

        if isinstance(self.clin_mlp, nn.Identity):
            clin_emb = None
        else:
            clin_emb = self.clin_mlp(clinical_feats)

        # FiLM
        if self.film is not None and lab_emb is not None:
            img_feats = self.film(img_feats, lab_emb)

        # Prepare features for gating and fusion
        current_img_feats = img_feats
        current_lab_emb = lab_emb
        current_clin_emb = clin_emb

        parts_for_gate = [current_img_feats]
        if current_lab_emb is not None: parts_for_gate.append(current_lab_emb)
        if current_clin_emb is not None: parts_for_gate.append(current_clin_emb)
        fused_in_for_gate = torch.cat(parts_for_gate, dim=-1)

        # gate weights (image/lab/clin)
        gate_w = self.gate(fused_in_for_gate)

        # Apply gating by scaling each component
        gated_components = []

        # Gated image features
        gated_img_feats = current_img_feats * gate_w[:, 0:1]
        gated_components.append(gated_img_feats)

        # Gated lab features
        if current_lab_emb is not None:
            gated_lab_emb = current_lab_emb * gate_w[:, 1:2]
            gated_components.append(gated_lab_emb)

        # Gated clinical features
        if current_clin_emb is not None:
            gated_clin_emb = current_clin_emb * gate_w[:, 2:3]
            gated_components.append(gated_clin_emb)

        # Concatenate the gated components for the fusion layer
        fused = self.fusion(torch.cat(gated_components, dim=-1))
        drug_logits = self.drug_head(fused)
        step_logits = self.step_head(fused)
        return drug_logits, step_logits, gate_w

lab_dim = len(LAB_COLS)
clin_dim = len(CLIN_COLS)

model = GC_MuPeN(lab_in_dim=lab_dim, clinical_in_dim=clin_dim,
                num_drugs=num_drugs, num_steps=num_steps).to(device)

print("Model ready.")

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Model ready.


## 9) Losses + Clinical safety regularizer (GCR)
GCR is **scaled** with `.mean()` to prevent exploding loss.


In [ ]:

# Class weights (clamped to avoid extreme values)
drug_counts = train_df[DRUG_COL].map(drug2id).value_counts().reindex(range(num_drugs), fill_value=0).values
drug_w = (drug_counts.sum() / (drug_counts + 1e-6))
drug_w = torch.tensor(drug_w, dtype=torch.float32, device=device)
drug_w = torch.clamp(drug_w, 0.2, 5.0)

step_counts = train_df["step_id"].value_counts().reindex(range(num_steps), fill_value=0).values
step_w = (step_counts.sum() / (step_counts + 1e-6))
step_w = torch.tensor(step_w, dtype=torch.float32, device=device)
step_w = torch.clamp(step_w, 0.2, 5.0)

ce_drug = nn.CrossEntropyLoss(weight=drug_w, label_smoothing=0.05)
ce_step = nn.CrossEntropyLoss(weight=step_w, label_smoothing=0.05)

# -------- GCR rules (conservative) --------
# These are soft penalties to discourage implausible Step↔Drug combos.
# We'll build a mask [num_steps, num_drugs] where 1 indicates "allowed".
allowed = torch.zeros((num_steps, num_drugs), dtype=torch.float32)

# Step 1: should mainly be H1_ANTIHISTAMINE
for d, idx in drug2id.items():
    if d.upper() == "H1_ANTIHISTAMINE":
        allowed[0, idx] = 1.0
    else:
        allowed[0, idx] = 0.4  # not impossible but discouraged

# Step 2: add-ons -> LTRA / OTHER / sometimes H1 (up-dosing)
for d, idx in drug2id.items():
    if d.upper() in ["LTRA", "OTHER", "H1_ANTIHISTAMINE"]:
        allowed[1, idx] = 1.0
    elif d.upper() == "ADVANCED_THERAPY":
        allowed[1, idx] = 0.2

# Step 3: potent antihistamine -> OTHER (common), maybe H1
for d, idx in drug2id.items():
    if d.upper() in ["OTHER", "H1_ANTIHISTAMINE"]:
        allowed[2, idx] = 1.0
    elif d.upper() == "ADVANCED_THERAPY":
        allowed[2, idx] = 0.2
    else:
        allowed[2, idx] = 0.6

# Step 4: advanced therapy -> ADVANCED_THERAPY
for d, idx in drug2id.items():
    if d.upper() == "ADVANCED_THERAPY":
        allowed[3, idx] = 1.0
    else:
        allowed[3, idx] = 0.2

allowed = allowed.to(device)

def guideline_consistency_penalty(step_logits, drug_logits):
    # Encourage high probability mass on "allowed" drug groups for the predicted step
    step_p = torch.softmax(step_logits, dim=-1)          # [B, S]
    drug_p = torch.softmax(drug_logits, dim=-1)          # [B, D]
    # expected allowed mass = sum_s sum_d step_p[s] * drug_p[d] * allowed[s,d]
    exp_allowed = torch.einsum("bs,bd,sd->b", step_p, drug_p, allowed)
    # penalty is (1 - allowed_mass); mean keeps scale stable
    return (1.0 - exp_allowed).mean()

# Training hyperparams
CONFIG = {
    "epochs": 50,
    "freeze_backbone_epochs": 5,
    "lr": 3e-4,
    "backbone_lr_factor": 0.1,
    "weight_decay": 1e-4,
    "step_loss_weight": 0.3,   # IMPORTANT (aux task)
    "gcr_lambda": 0.01,        # small & safe
    "warmup_epochs": 2,
}

print(CONFIG)


{'epochs': 50, 'freeze_backbone_epochs': 5, 'lr': 0.0003, 'backbone_lr_factor': 0.1, 'weight_decay': 0.0001, 'step_loss_weight': 0.3, 'gcr_lambda': 0.01, 'warmup_epochs': 2}


## 10) Optimizer + Warmup Cosine scheduler

In [ ]:

def build_optimizer(model):
    backbone_params = []
    other_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "image_backbone" in name:
            backbone_params.append(p)
        else:
            other_params.append(p)

    optimizer = optim.AdamW(
        [
            {"params": backbone_params, "lr": CONFIG["lr"] * CONFIG["backbone_lr_factor"]},
            {"params": other_params, "lr": CONFIG["lr"]},
        ],
        weight_decay=CONFIG["weight_decay"],
    )
    return optimizer

optimizer = build_optimizer(model)

total_steps = CONFIG["epochs"] * len(train_loader)

def lr_lambda(current_step):
    # warmup then cosine
    warmup_steps = CONFIG["warmup_epochs"] * len(train_loader)
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


## 11) Train + Eval loops

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total = 0
    correct_drug = 0
    correct_step = 0
    losses = []

    all_y = []
    all_pred = []

    for batch in loader:
        image = batch["image"].to(device)
        lab = batch["lab_feats"].to(device)
        clin = batch["clinical_feats"].to(device)
        y_drug = batch["drug_label"].to(device)
        y_step = batch["step_label"].to(device)

        drug_logits, step_logits, _ = model(image, lab, clin)

        loss_drug = ce_drug(drug_logits, y_drug)
        loss_step = ce_step(step_logits, y_step)
        gcr = guideline_consistency_penalty(step_logits, drug_logits)

        loss = loss_drug + CONFIG["step_loss_weight"]*loss_step + CONFIG["gcr_lambda"]*gcr
        losses.append(loss.item())

        pred_drug = drug_logits.argmax(dim=-1)
        pred_step = step_logits.argmax(dim=-1)

        total += y_drug.size(0)
        correct_drug += (pred_drug == y_drug).sum().item()
        correct_step += (pred_step == y_step).sum().item()

        all_y.append(y_drug.detach().cpu().numpy())
        all_pred.append(pred_drug.detach().cpu().numpy())

    acc_drug = correct_drug / max(1, total)
    acc_step = correct_step / max(1, total)
    return float(np.mean(losses)), acc_drug, acc_step, np.concatenate(all_y), np.concatenate(all_pred)

def train_one_epoch(epoch, global_step):
    model.train()
    total = 0
    correct_drug = 0
    correct_step = 0
    losses = []

    for batch in train_loader:
        image = batch["image"].to(device)
        lab = batch["lab_feats"].to(device)
        clin = batch["clinical_feats"].to(device)
        y_drug = batch["drug_label"].to(device)
        y_step = batch["step_label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        drug_logits, step_logits, _ = model(image, lab, clin)

        loss_drug = ce_drug(drug_logits, y_drug)
        loss_step = ce_step(step_logits, y_step)
        gcr = guideline_consistency_penalty(step_logits, drug_logits)

        loss = loss_drug + CONFIG["step_loss_weight"]*loss_step + CONFIG["gcr_lambda"]*gcr

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        global_step += 1

        losses.append(loss.item())

        pred_drug = drug_logits.argmax(dim=-1)
        pred_step = step_logits.argmax(dim=-1)

        total += y_drug.size(0)
        correct_drug += (pred_drug == y_drug).sum().item()
        correct_step += (pred_step == y_step).sum().item()

    return float(np.mean(losses)), correct_drug/max(1,total), correct_step/max(1,total), global_step

history = {
    "train_loss": [], "val_loss": [],
    "train_drug_acc": [], "val_drug_acc": [],
    "train_step_acc": [], "val_step_acc": [],
}

# best_val = 0.0
# best_path = os.path.join(ARTIFACT_ROOT, "model.pt")

# global_step = 0

# --- add/replace these lines ---
best_val = 0.0
best_path = os.path.join(ARTIFACT_ROOT, "model.pt")

patience = int(CONFIG.get("patience", 10))
min_delta = float(CONFIG.get("min_delta", 1e-4))
epochs_no_improve = 0

global_step = 0

print("="*70)
print("Starting training GC-MuPeN v3")
print("="*70)

for epoch in range(1, CONFIG["epochs"]+1):
    if epoch == CONFIG["freeze_backbone_epochs"] + 1:
        print("="*70)
        print("UNFREEZING BACKBONE AT EPOCH", epoch)
        print("="*70)
        model.unfreeze_backbone()
        optimizer = build_optimizer(model)  # rebuild with backbone params
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    tr_loss, tr_dacc, tr_sacc, global_step = train_one_epoch(epoch, global_step)
    va_loss, va_dacc, va_sacc, _, _ = evaluate(val_loader)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_drug_acc"].append(tr_dacc)
    history["val_drug_acc"].append(va_dacc)
    history["train_step_acc"].append(tr_sacc)
    history["val_step_acc"].append(va_sacc)

    print(f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
          f"TrainLoss {tr_loss:.4f} DrugAcc {tr_dacc*100:.2f}% StepAcc {tr_sacc*100:.2f}% || "
          f"ValLoss {va_loss:.4f} DrugAcc {va_dacc*100:.2f}% StepAcc {va_sacc*100:.2f}%")

    

        # --- replace your "best model save" block with this ---
    if va_dacc > best_val + min_delta:
        best_val = va_dacc
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ New best saved! Val Drug Acc: {best_val*100:.2f}%")
    else:
        epochs_no_improve += 1
        print(f"  ⏳ EarlyStopping counter: {epochs_no_improve}/{patience}")

        if epochs_no_improve >= patience:
            print(f"  🛑 Early stopping triggered (no Val Drug Acc improvement for {patience} epochs).")
            break

# --- add after the loop ---
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
print("Best Val Drug Acc:", best_val*100)


Starting training GC-MuPeN v3
Epoch 01/50 | TrainLoss 1.5369 DrugAcc 34.43% StepAcc 30.75% || ValLoss 1.7590 DrugAcc 25.41% StepAcc 23.14%
  ✓ New best saved! Val Drug Acc: 25.41%
Epoch 02/50 | TrainLoss 1.1547 DrugAcc 56.06% StepAcc 46.61% || ValLoss 1.4105 DrugAcc 70.04% StepAcc 48.76%
  ✓ New best saved! Val Drug Acc: 70.04%
Epoch 03/50 | TrainLoss 1.0852 DrugAcc 63.34% StepAcc 55.19% || ValLoss 1.7159 DrugAcc 48.97% StepAcc 40.91%
  ⏳ EarlyStopping counter: 1/10
Epoch 04/50 | TrainLoss 1.0285 DrugAcc 65.66% StepAcc 57.76% || ValLoss 1.4041 DrugAcc 61.36% StepAcc 53.31%
  ⏳ EarlyStopping counter: 2/10
Epoch 05/50 | TrainLoss 0.9850 DrugAcc 68.04% StepAcc 61.15% || ValLoss 1.4680 DrugAcc 58.88% StepAcc 54.96%
  ⏳ EarlyStopping counter: 3/10
UNFREEZING BACKBONE AT EPOCH 6
Epoch 06/50 | TrainLoss 0.9569 DrugAcc 69.74% StepAcc 61.15% || ValLoss 1.3851 DrugAcc 61.16% StepAcc 54.96%
  ⏳ EarlyStopping counter: 4/10
Epoch 07/50 | TrainLoss 0.9193 DrugAcc 70.17% StepAcc 62.85% || ValLoss 1.3

## 12) Test set evaluation + reports

In [ ]:

# Load best weights
model.load_state_dict(torch.load(best_path, map_location=device))

test_loss, test_dacc, test_sacc, y_true, y_pred = evaluate(test_loader)
print(f"TEST | Loss {test_loss:.4f} DrugAcc {test_dacc*100:.2f}% StepAcc {test_sacc*100:.2f}%")

print("\nDrug Group Classification Report:")
print(classification_report(y_true, y_pred, target_names=[id2drug[i] for i in range(num_drugs)]))


TEST | Loss 1.2875 DrugAcc 73.35% StepAcc 75.55%

Drug Group Classification Report:
                  precision    recall  f1-score   support

ADVANCED_THERAPY       0.58      0.58      0.58        64
H1_ANTIHISTAMINE       0.89      0.92      0.90       230
            LTRA       0.38      0.20      0.26        46
           OTHER       0.59      0.66      0.62       114

        accuracy                           0.73       454
       macro avg       0.61      0.59      0.59       454
    weighted avg       0.72      0.73      0.72       454



## 13) Temperature scaling (calibration) + abstain
This saves `temperature.json` for backend.


In [ ]:

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        t = torch.exp(self.log_t).clamp(1e-2, 100.0)
        return logits / t

@torch.no_grad()
def collect_logits(loader):
    model.eval()
    all_logits = []
    all_labels = []
    for batch in loader:
        image = batch["image"].to(device)
        lab = batch["lab_feats"].to(device)
        clin = batch["clinical_feats"].to(device)
        y = batch["drug_label"].to(device)
        logits, _, _ = model(image, lab, clin)
        all_logits.append(logits)
        all_labels.append(y)
    return torch.cat(all_logits), torch.cat(all_labels)

val_logits, val_labels = collect_logits(val_loader)

scaler = TemperatureScaler().to(device)
criterion = nn.CrossEntropyLoss()

opt = optim.LBFGS(scaler.parameters(), lr=0.5, max_iter=50)

def closure():
    opt.zero_grad()
    loss = criterion(scaler(val_logits), val_labels)
    loss.backward()
    return loss

opt.step(closure)

temp = float(torch.exp(scaler.log_t).detach().cpu().item())
print("Learned temperature:", temp)

with open(os.path.join(ARTIFACT_ROOT, "temperature.json"), "w") as f:
    json.dump({"temperature": temp}, f, indent=2)


Learned temperature: 1.4646079540252686


## 14) Save backend artifacts (`config.json`, `history.json`)
Backend uses these files to reproduce preprocessing + label mapping.


In [ ]:

config = {
    "image_size": IMG_SIZE,
    "drug_groups": drug_groups,
    "guideline_steps": ["STEP_1","STEP_2","STEP_3","STEP_4"],
    "lab_features": LAB_COLS,
    "clinical_features": CLIN_COLS,
    "lab_means": lab_means.tolist() if len(LAB_COLS)>0 else [],
    "lab_stds": lab_stds.tolist() if len(LAB_COLS)>0 else [],
    "clinical_means": clin_means.tolist() if len(CLIN_COLS)>0 else [],
    "clinical_stds": clin_stds.tolist() if len(CLIN_COLS)>0 else [],
    "image_col": IMAGE_COL,
    "drug_col": DRUG_COL,
    "split_col": SPLIT_COL,
    "drug_name_col": DRUG_NAME_COL,
    "model_name": "GC_MuPeN_v3_EfficientNetB3_FiLM_GatedFusion",
}

with open(os.path.join(ARTIFACT_ROOT, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

with open(os.path.join(ARTIFACT_ROOT, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

print("Saved artifacts to:", ARTIFACT_ROOT)
print(" - model.pt")
print(" - config.json")
print(" - history.json")
print(" - temperature.json")


Saved artifacts to: /content/drive/MyDrive/AURA_artifacts/gcmupen_v3_final/weights
 - model.pt
 - config.json
 - history.json
 - temperature.json


## 15) Backend inference helper (FastAPI-ready)
Copy this into your backend to load `model.pt + config.json + temperature.json`.


In [ ]:

# --- Backend-style inference snippet (runs in Colab too) ---
import torch.nn.functional as F

def load_artifacts(weights_dir: str):
    with open(os.path.join(weights_dir, "config.json"), "r") as f:
        cfg = json.load(f)
    with open(os.path.join(weights_dir, "temperature.json"), "r") as f:
        tcfg = json.load(f)

    drug_groups = cfg["drug_groups"]
    lab_cols = cfg["lab_features"]
    clin_cols = cfg["clinical_features"]

    lab_means = np.array(cfg["lab_means"], dtype=np.float32) if len(lab_cols)>0 else np.array([])
    lab_stds  = np.array(cfg["lab_stds"], dtype=np.float32) if len(lab_cols)>0 else np.array([])
    clin_means = np.array(cfg["clinical_means"], dtype=np.float32) if len(clin_cols)>0 else np.array([])
    clin_stds  = np.array(cfg["clinical_stds"], dtype=np.float32) if len(clin_cols)>0 else np.array([])

    model = GC_MuPeN(
        lab_in_dim=len(lab_cols),
        clinical_in_dim=len(clin_cols),
        num_drugs=len(drug_groups),
        num_steps=4
    ).to(device)

    sd = torch.load(os.path.join(weights_dir, "model.pt"), map_location=device)
    model.load_state_dict(sd)
    model.eval()

    temperature = float(tcfg["temperature"])
    return cfg, model, temperature, lab_means, lab_stds, clin_means, clin_stds

cfg, infer_model, temperature, lmean, lstd, cmean, cstd = load_artifacts(ARTIFACT_ROOT)

def preprocess_inputs(image_pil: Image.Image, lab_dict: Dict[str,float], clin_dict: Dict[str,float], cfg):
    tfm = get_transforms(False)
    img = tfm(image_pil.convert("RGB")).unsqueeze(0).to(device)

    lab_cols = cfg["lab_features"]
    clin_cols = cfg["clinical_features"]

    if len(lab_cols)>0:
        lab = np.array([float(lab_dict.get(k, 0.0)) for k in lab_cols], dtype=np.float32)
        lab = (lab - lmean) / lstd
        lab = torch.tensor(lab, dtype=torch.float32).unsqueeze(0).to(device)
    else:
        lab = torch.zeros((1,0), device=device)

    if len(clin_cols)>0:
        clin = np.array([float(clin_dict.get(k, 0.0)) for k in clin_cols], dtype=np.float32)
        clin = (clin - cmean) / cstd
        clin = torch.tensor(clin, dtype=torch.float32).unsqueeze(0).to(device)
    else:
        clin = torch.zeros((1,0), device=device)

    return img, lab, clin

@torch.no_grad()
def predict(image_pil, lab_dict, clin_dict, abstain_threshold=0.55):
    img, lab, clin = preprocess_inputs(image_pil, lab_dict, clin_dict, cfg)
    drug_logits, step_logits, gate_w = infer_model(img, lab, clin)

    drug_logits = drug_logits / temperature
    probs = F.softmax(drug_logits, dim=-1).squeeze(0).cpu().numpy()

    top_idx = int(np.argmax(probs))
    top_conf = float(probs[top_idx])

    step = int(torch.argmax(step_logits, dim=-1).item()) + 1  # 1..4

    result = {
        "predicted_drug_group": cfg["drug_groups"][top_idx],
        "confidence": top_conf,
        "top3": sorted([(cfg["drug_groups"][i], float(probs[i])) for i in range(len(probs))],
                       key=lambda x: x[1], reverse=True)[:3],
        "mapped_guideline_step": f"STEP_{step}",
        "abstain": bool(top_conf < abstain_threshold),
        "modality_gate_weights": gate_w.squeeze(0).detach().cpu().numpy().tolist()
    }
    return result

print("Backend helper ready. Example usage:")
print("predict(Image.open('/path/to/img.jpg'), lab_dict={}, clin_dict={})")


Backend helper ready. Example usage:
predict(Image.open('/path/to/img.jpg'), lab_dict={}, clin_dict={})
